In [2]:
import pandas as pd
import tensorflow as tf
from tensorflow import  keras # type: ignore
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder()
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier

In [3]:
df = pd.read_csv('..\data\data_clean.csv')

In [4]:
df .head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,0,2
1,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,0,3
2,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,0,2
3,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,1,4
4,21,9900,OWN,2.0,VENTURE,A,2500,7.14,1,0.25,0,2


In [5]:
df_encoder = encoder.fit_transform(df[['loan_grade','loan_intent','person_home_ownership']])

In [6]:
encoded_df = pd.DataFrame(
    df_encoder.toarray(),
    columns=encoder.get_feature_names_out(['loan_grade','loan_intent','person_home_ownership']),
    index=df.index
)

In [7]:
df = pd.concat([df.drop(columns=['loan_grade','loan_intent','person_home_ownership']),
                encoded_df],
               axis=1)

In [8]:
df

,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length,loan_grade_A,...,loan_intent_DEBTCONSOLIDATION,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE,person_home_ownership_MORTGAGE,person_home_ownership_OTHER,person_home_ownership_OWN,person_home_ownership_RENT
0,21,9600,5.0,1000,11.14,0,0.10,0,2,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,25,9600,1.0,5500,12.87,1,0.57,0,3,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
2,23,65500,4.0,35000,15.23,1,0.53,0,2,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
3,24,54400,8.0,35000,14.27,1,0.55,1,4,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
4,21,9900,2.0,2500,7.14,1,0.25,0,2,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32569,57,53000,1.0,5800,13.16,0,0.11,0,30,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
32570,54,120000,4.0,17625,7.49,0,0.15,0,19,1.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
32571,65,76000,3.0,35000,10.99,1,0.46,0,28,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
32572,56,150000,5.0,15000,11.48,0,0.10,0,26,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0


In [9]:
X = df.drop(columns=['loan_status'])
y = df['loan_status']

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=100)


## LogisticRegression

In [10]:
LO = LogisticRegression(random_state=0,max_iter=100)
LO.fit(X_train,y_train)

LogisticRegression(random_state=0)

In [11]:
predictions = LO.predict(X_test)
acc = accuracy_score(y_test, predictions) * 100
print(f"Logistic Regression model accuracy: {acc:.2f}%")

Logistic Regression model accuracy: 81.06%


In [12]:
report = classification_report(y_test,predictions)
print(report)

              precision    recall  f1-score   support

           0       0.81      0.99      0.89      5133
           1       0.75      0.16      0.26      1382

    accuracy                           0.81      6515
   macro avg       0.78      0.57      0.58      6515
weighted avg       0.80      0.81      0.76      6515



## XGBClassifier

In [13]:
xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss',
    scale_pos_weight=3.71
)
xgb.fit(X_train,y_train)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=500, n_jobs=None,
              num_parallel_tree=None, ...)

In [14]:
xgb_predictions = xgb.predict(X_test)
xgb_acc = accuracy_score(y_test, xgb_predictions) * 100

In [15]:
report_xgb = classification_report(y_test,xgb_predictions)

In [16]:
print(report_xgb)

              precision    recall  f1-score   support

           0       0.95      0.95      0.95      5133
           1       0.81      0.81      0.81      1382

    accuracy                           0.92      6515
   macro avg       0.88      0.88      0.88      6515
weighted avg       0.92      0.92      0.92      6515



In [17]:
probs = xgb.predict_proba(X_test)[:, 1]
print(probs)

[0.49666145 0.12438804 0.39218566 ... 0.00116036 0.09043339 0.3162376 ]


In [18]:
threshold = 0.4
new_predictions = (probs >= threshold).astype(int)

In [19]:
new_predictions

array([1, 0, 0, ..., 0, 0, 0])

In [20]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_test, new_predictions))
print(confusion_matrix(y_test, new_predictions))

              precision    recall  f1-score   support

           0       0.96      0.89      0.92      5133
           1       0.67      0.86      0.76      1382

    accuracy                           0.88      6515
   macro avg       0.82      0.88      0.84      6515
weighted avg       0.90      0.88      0.89      6515

[[4556  577]
 [ 189 1193]]


## Random forest

In [21]:

rfc = RandomForestClassifier(n_estimators=100, random_state=42)
rfc.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [22]:
rfc_prediction = rfc.predict(X_test)

In [23]:
print(classification_report(y_test, rfc_prediction))

              precision    recall  f1-score   support

           0       0.93      0.99      0.96      5133
           1       0.97      0.74      0.84      1382

    accuracy                           0.94      6515
   macro avg       0.95      0.87      0.90      6515
weighted avg       0.94      0.94      0.94      6515



## Balanced data

In [24]:
df0 = df[df['loan_status'] == 0]
df1 = df[df['loan_status'] == 1]

In [25]:
df0

,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length,loan_grade_A,...,loan_intent_DEBTCONSOLIDATION,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE,person_home_ownership_MORTGAGE,person_home_ownership_OTHER,person_home_ownership_OWN,person_home_ownership_RENT
0,21,9600,5.0,1000,11.14,0,0.10,0,2,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
13,23,115000,2.0,35000,7.90,0,0.30,0,4,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
14,23,500000,7.0,30000,10.65,0,0.06,0,3,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
15,23,120000,0.0,35000,7.90,0,0.29,0,4,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
19,25,162500,2.0,35000,7.49,0,0.22,0,4,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32568,52,64500,0.0,5000,11.26,0,0.08,0,20,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
32569,57,53000,1.0,5800,13.16,0,0.11,0,30,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
32570,54,120000,4.0,17625,7.49,0,0.15,0,19,1.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
32572,56,150000,5.0,15000,11.48,0,0.10,0,26,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0


In [26]:
df0_sampled = df0.sample(n=len(df1),random_state=42)

In [27]:
df_balnced = pd.concat([df0_sampled,df1],axis=0)

In [28]:
df_balnced['loan_status'].value_counts()

loan_status
0    7107
1    7107
Name: count, dtype: int64

In [29]:
X = df_balnced.drop(columns=['loan_status'])
y = df_balnced['loan_status']

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=100)


In [30]:
xgb = XGBClassifier(
    n_estimators=800,
    learning_rate=0.05,
    max_depth=10,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss',
    
)
xgb.fit(X_train,y_train)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=10, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=800, n_jobs=None,
              num_parallel_tree=None, ...)

In [31]:
xgb_predictions = xgb.predict(X_test)
xgb_acc = accuracy_score(y_test, xgb_predictions) * 100

In [32]:
report_xgb = classification_report(y_test,xgb_predictions)

In [33]:
print(report_xgb)

              precision    recall  f1-score   support

           0       0.84      0.91      0.87      1416
           1       0.90      0.83      0.86      1427

    accuracy                           0.87      2843
   macro avg       0.87      0.87      0.87      2843
weighted avg       0.87      0.87      0.87      2843



## ROC-AUC

In [34]:
probs = xgb.predict_proba(X_test)[:,1]
roc_auc_score(y_test, probs)

0.946862169855768

## save model

In [35]:
xgb.save_model("xgb_model.pickle")

c:\Users\Amr Khaled\AppData\Local\Programs\Python\Python310\lib\site-packages\xgboost\sklearn.py:1028: UserWarning: [00:20:28] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  self.get_booster().save_model(fname)
